# Classification

In [ ]:
import typing

import numpy as np
import numpy.typing as npt
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, classification_report, confusion_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Метрики классификации

### Accuracy

Доля верных ответов.  

$$ Accuracy = \frac{ N_{correct} } {N} = \frac{ TP + TN } { TP + TN + FP + FN } $$

### Precision:

Точность (доля) положительных предсказаний.  
Метрика тем хуже, чем больше ошибок мы допустили в нашем положительном предсказании.

$$ Precision = \frac{ TP } { TP + FP } $$

### Recall:

Полнота обнаружения положительного класса.  
Метрика тем хуже, чем больше положительных меток мы предсказали неверно (пропустили).

$$ Recall = \frac{ TP } { TP + FN } $$

### F1:

Гармоническое среднее precision и recall.

$$ {"F1-score"} = \frac{ 2 * Precision * Recall } { Precision + Recall } $$

In [ ]:
# todo: implement metrics

In [ ]:
def report_classification_metrics(y_true, y_pred):
    print(f"Accuracy: {accuracy(y_true, y_pred):.2f}")
    print(f"Precision: {precision(y_true, y_pred):.2f}")
    print(f"Recall: {recall(y_true, y_pred):.2f}")
    print(f"F1-Score: {f1(y_true, y_pred):.2f}")

## Bayes classifiers

In [ ]:
# Сгенерируем датасет
X, y = make_blobs(
    n_samples=400,
    centers=2,
    cluster_std=3,
    random_state=42
)

df = pl.DataFrame({
    "x1": X[:, 0],
    "x2": X[:, 1],
    "y": y
})
df.head()

In [ ]:
plt.scatter(df["x1"], df["x2"], c=df["y"])

### Gaussian Naive Bayes

Формула Байеса:

$$ P(y|x) = P(y) \prod P(x_j | y) $$

и предполагаем:

$$ P(x_j | y) \sim N(μ, σ²)$$

In [ ]:
# todo: implement GaussianNB from scratch

посмотрим насколько это похоже на реализацию в пакете sklearn: [документация](https://scikit-learn.org/stable/modules/naive_bayes.html#gaussian-naive-bayes), [код](https://github.com/scikit-learn/scikit-learn/blob/d3898d9d5/sklearn/naive_bayes.py#L163)

**Вопрос**: почему в sklearn вероятности считаются через логарифмы?

$$ \log { P(y|x) } \propto \log { P(y) } + \sum_p \log { P(x_j | y) } $$

In [ ]:
# sklearn simple example

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

classifier = GaussianNB()
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

report_classification_metrics(y_test, y_pred)

In [ ]:
def plot_decision_boundary(classifier, x1, x2, y):
    x_axis_min, x_axis_max = x1.min(), x1.max()
    y_axis_min, y_axis_max = x2.min(), x2.max()

    xx, yy = np.meshgrid(
        np.linspace(x_axis_min, x_axis_max, 100),
        np.linspace(y_axis_min, y_axis_max, 100))

    Z = classifier.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, cmap="summer", alpha=0.5)
    plt.scatter(x1, x2, c=y)
    plt.show()

In [ ]:
plot_decision_boundary(classifier, X[:, 0], X[:, 1], y)

In [ ]:
my_classifier = GaussianNBManual()
my_classifier.fit(X_train, y_train)
y_pred = my_classifier.predict(X_test)

report_classification_metrics(y_test, y_pred)

In [ ]:
y_proba_manual = classifier.predict_proba(X_test)
y_proba_sklearn = my_classifier.predict_proba(X_test)

prob_diff = np.mean(np.abs(y_proba_manual - y_proba_sklearn))
print(f"Mean absolute probability difference: {prob_diff:.6f}")

#### Классификация спама


In [ ]:
df = pl.read_csv('./spam_ham_dataset.csv')
df.head()

In [ ]:
import string
from wordcloud import STOPWORDS


class TextPreprocessor:
    def __init__(self):
        self.stopwords = STOPWORDS.union(set(['subject', 're', 'enron']))

    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        processed_X = []

        todo = [
            self._lowercase,
            self._remove_punctuation,
            self._remove_stopwords,
        ]

        for text in X:
            for f in todo:
                text = f(text)
            processed_X.append(text)
        
        return np.array(processed_X)

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X, y)

    def _lowercase(self, text):
        return text.lower()

    def _remove_punctuation(self, text):
        for punctuation in string.punctuation:
            text = text.replace(punctuation, '')
        return text

    def _remove_stopwords(self, text):
        new_text = []
        for word in text.split():
            if word in self.stopwords:
                continue
            new_text.append(word)
        return ' '.join(new_text)

In [ ]:
X = df['text'].to_numpy()
y = df['label_num'].to_numpy()


X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y,
)

pipeline = Pipeline([
    ('text_preprocessor', TextPreprocessor()),
    ('vectorizer', CountVectorizer()),
    ('classifier', MultinomialNB(
        alpha=1.0,
        fit_prior=True,
    ))
])

In [ ]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

## Logistic regression

In [ ]:
df_raw = pl.read_csv('breast-cancer.csv')
df_raw.head()

In [ ]:
df_raw.get_column("diagnosis").value_counts()

In [ ]:
df = df_raw.with_columns(
    pl.col("diagnosis")
      .replace("B", "0")
      .replace("M", "1")
      .cast(pl.Int64)
      .alias("target")
).drop('diagnosis')
df.describe()

In [ ]:
TARGET_COLUMN_NAME = "target"

df.get_column(TARGET_COLUMN_NAME).value_counts()

In [ ]:
X, y = df.drop(TARGET_COLUMN_NAME).to_numpy(), df.get_column(TARGET_COLUMN_NAME).to_numpy()

X_test, X_train, y_test, y_train = train_test_split(
    X,
    y,
    test_size=0.5,
    random_state=42
)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(
        penalty='l2',
        C=0.1,
    ))
])

In [ ]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))

### ROC-curve

In [ ]:
y_scores = pipeline.predict_proba(X_test)[:,1]

fpr, tpr, thresholds = roc_curve(y_test, y_scores)
roc_auc = auc(fpr, tpr)

print(f"ROC AUC: {roc_auc}")

In [ ]:
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.4f})", color='orange')
plt.plot([0, 1], [0, 1], 'k--', color='grey')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic - Breast Cancer Logistic Regression')
plt.legend(loc="lower right")
plt.show()

### Задача:
Подобрать такой threshold вероятности, чтобы удовлетворять следующему требованию:

$$ TPR (Recall) \ge 99\% $$

То есть требуется с высокой долей уверенности (99% - "2 девятки") обнаруживать рак груди у пациентов, т.к. цена ошибки высока.

In [ ]:
print(f"Current recall score: {recall(y_test, y_pred):%}")
print(f"Current F1 score: {f1(y_test, y_pred):.4f}")

In [ ]:
target_min_tpr = 0.99

best_index = np.argmin(fpr[tpr > target_min_tpr])
best_threshold = thresholds[tpr > target_min_tpr][best_index]

print(f"FPR: {fpr[tpr > target_min_tpr][best_index]:.4f}")
print(f"TPR: {tpr[tpr > target_min_tpr][best_index]:.4f}")
print(f"Threshold: {best_threshold:.4f}")

In [ ]:
y_pred_new = (y_scores >= best_threshold).astype(int)

print("Before tweaking threshold:")
report_classification_metrics(y_test, y_pred)

print("\nWith tweaked threshold:")
report_classification_metrics(y_test, y_pred_new)

## One vs All classifier

In [ ]:
X, y = make_blobs(
    n_samples=500,
    centers=3,
    cluster_std=2,
    random_state=42,
)

In [ ]:
plt.scatter(X[:,0], X[:,1], c=y)

In [ ]:
class OneVsAllClassifier:
    def __init__(self, base_clf):
        self.base_clf = base_clf

    def fit(self, X, y):
        self.classes = np.unique(y)
        self.models = {}

        for c in self.classes:
            y_bin = (y == c).astype(int)
            clf = self.base_clf()
            clf.fit(X, y_bin)
            self.models[c] = clf

    def predict(self, X):
        scores = np.column_stack([
            self.models[c].predict_proba(X)[:,1]
            for c in self.classes
        ])
        return np.argmax(scores, axis=1)

In [ ]:
def new_classifier():
    return LogisticRegression()

classifier = OneVsAllClassifier(new_classifier)
classifier.fit(X, y)

plot_decision_boundary(classifier, X[:, 0], X[:, 1], y)